# Scaling-Law Analysis

End-to-end analysis of the SP and µP runs. Produces the figures cited in `report/report.tex`.

**Inputs** (produced by the training and evaluation scripts):
- `checkpoints/{size}/log.csv` — per-step train/val loss, LR, elapsed seconds (one per SP run)
- `checkpoints/{size}_mup/log.csv` — same, µP runs
- `checkpoints/{size}/sweep/sweep_results.csv` — `(lr, best_val_loss)` per LR sweep
- `results/{size}_eval.json` — perplexity, XML rate, render rate per size

**Outputs** — figures saved to `report/figures/` for inclusion in the LaTeX report.

## 1. Setup

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 200})

FIG_DIR = Path('report/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Five model sizes — must match config filenames and N_non_emb computed in transformer.py
SIZES = ['1m', '3m', '12m', '34m', '88m']
N_NON_EMB = {
    '1m':  1_049_728,
    '3m':  3_148_032,
    '12m': 11_016_576,
    '34m': 35_668_480,
    '88m': 88_108_800,
}

CKPT_ROOT    = Path('checkpoints')
RESULTS_ROOT = Path('results')


def safe_read_csv(path):
    return pd.read_csv(path) if os.path.exists(path) else None

def safe_read_json(path):
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

## 2. Load all results

Robust to missing files — sections that depend on absent data simply skip rather than fail.

In [ ]:
sp_logs   = {s: safe_read_csv(CKPT_ROOT / s              / 'log.csv') for s in SIZES}
mup_logs  = {s: safe_read_csv(CKPT_ROOT / f'{s}_mup'     / 'log.csv') for s in SIZES}
sweeps     = {s: safe_read_csv(CKPT_ROOT / f'{s}_sweep'      / 'sweep_results.csv') for s in SIZES}
mup_sweeps = {s: safe_read_csv(CKPT_ROOT / f'{s}_mup_sweep'  / 'sweep_results.csv') for s in SIZES}
evals     = {s: safe_read_json(RESULTS_ROOT / f'{s}_eval.json') for s in SIZES}

available = {
    'SP train':  [s for s in SIZES if sp_logs[s]  is not None],
    'µP train':  [s for s in SIZES if mup_logs[s] is not None],
    'LR sweep':  [s for s in SIZES if sweeps[s]   is not None],
    'Eval':      [s for s in SIZES if evals[s]    is not None],
}
for k, v in available.items():
    print(f'{k:10s}: {v}')

## 3. Training curves

Train and val loss vs step for each model size. Two panels: SP on the left, µP on the right. Same y-axis so curves are visually comparable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

for ax, logs, title in [(axes[0], sp_logs, 'SP'), (axes[1], mup_logs, 'µP')]:
    for s in SIZES:
        df = logs[s]
        if df is None:
            continue
        ax.plot(df['step'], df['val_loss'], label=f'{s}  (N={N_NON_EMB[s]:,})')
    ax.set_xlabel('Step')
    ax.set_title(f'{title} validation loss')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_ylabel('Cross-entropy loss')

plt.tight_layout()
plt.savefig(FIG_DIR / 'training_curves.pdf')
plt.show()

## 4. LR sweep — choosing the optimal LR per size

For each size, val loss vs LR (log x). The minimum identifies the LR committed to that size's full-run config. Under µP we expect the optimum to transfer across sizes; under SP it should drift.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

for s in SIZES:
    df = sweeps[s]
    if df is None:
        continue
    df_sorted = df.sort_values('lr')
    ax.plot(df_sorted['lr'], df_sorted['best_val_loss'], 'o-', label=f'{s}')

ax.set_xscale('log')
ax.set_xlabel('Learning rate')
ax.set_ylabel('Best val loss (15% budget)')
ax.set_title('LR sweep — val loss vs LR per size')
ax.grid(alpha=0.3, which='both')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'lr_sweep.pdf')
plt.show()

## 5. Power-law fit:  L = a · N^(-α) + c

The headline scaling-law deliverable. Fit on the five $(N, L^*)$ points using `scipy.optimize.curve_fit` with positive bounds. Report $\alpha$, $c$, and 95% CIs derived from the covariance matrix.

In [ ]:
def power_law(N, a, alpha, c):
    return a * N ** (-alpha) + c

def fit_power_law(N_arr, L_arr):
    """Returns (popt, perr) where perr is 1-sigma uncertainty per parameter."""
    popt, pcov = curve_fit(
        power_law, N_arr, L_arr,
        p0=[1.0, 0.1, min(L_arr)],
        bounds=([0, 0, 0], [np.inf, 1.0, np.inf]),
        maxfev=10000,
    )
    perr = np.sqrt(np.diag(pcov))
    return popt, perr


def collect_final_loss(logs):
    """For each size, take the best val loss across the run (matches what train.py saves as best.pt)."""
    return {s: float(logs[s]['val_loss'].min()) for s in SIZES if logs[s] is not None}


sp_finals  = collect_final_loss(sp_logs)
mup_finals = collect_final_loss(mup_logs)

summary = pd.DataFrame({
    'N':       [N_NON_EMB[s]   for s in SIZES],
    'SP_loss': [sp_finals.get(s, np.nan)  for s in SIZES],
    'µP_loss': [mup_finals.get(s, np.nan) for s in SIZES],
}, index=SIZES)
summary

In [ ]:
fits = {}
for label, finals in [('SP', sp_finals), ('µP', mup_finals)]:
    if len(finals) < 3:
        print(f'{label}: not enough points to fit ({len(finals)} < 3)')
        continue
    sizes_fit = [s for s in SIZES if s in finals]
    N = np.array([N_NON_EMB[s] for s in sizes_fit])
    L = np.array([finals[s]    for s in sizes_fit])
    (a, alpha, c), (a_err, alpha_err, c_err) = fit_power_law(N, L)
    fits[label] = {'a': a, 'alpha': alpha, 'c': c,
                   'a_err': a_err, 'alpha_err': alpha_err, 'c_err': c_err,
                   'N': N, 'L': L}
    print(f'{label}:  L = {a:.3g} · N^(-{alpha:.4f}) + {c:.4f}')
    print(f'       α = {alpha:.4f} ± {1.96*alpha_err:.4f}   c = {c:.4f} ± {1.96*c_err:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

N_smooth = np.logspace(6, 9.5, 200)   # 1M to ~3B for extrapolation visualization

for label, color in [('SP', 'C0'), ('µP', 'C1')]:
    if label not in fits:
        continue
    f = fits[label]
    ax.scatter(f['N'], f['L'], color=color, label=f'{label} measured', zorder=3)
    ax.plot(N_smooth, power_law(N_smooth, f['a'], f['alpha'], f['c']),
            color=color, linestyle='--', alpha=0.7,
            label=f'{label} fit:  α={f["alpha"]:.3f}, c={f["c"]:.3f}')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Non-embedding parameters $N$')
ax.set_ylabel('Validation loss $L^*$')
ax.set_title('Scaling law:  $L = a \\cdot N^{-\\alpha} + c$')
ax.grid(alpha=0.3, which='both')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'scaling_law.pdf')
plt.show()

## 6. µP extrapolation

Use the µP fit to predict val loss at ~10× the largest fitted size (~880M params). If a held-out run at that scale exists, plot it as a measured point to verify the prediction.

In [ ]:
if 'µP' in fits:
    f = fits['µP']
    N_target = 10 * N_NON_EMB['88m']
    L_pred   = power_law(N_target, f['a'], f['alpha'], f['c'])
    print(f'µP extrapolation:')
    print(f'  Target N = {N_target:,}  (10× the largest fitted)')
    print(f'  Predicted val loss = {L_pred:.4f}')
    print(f'  Predicted perplexity = {np.exp(L_pred):.2f}')
else:
    print('No µP fit available yet.')

## 7. Sample quality:  XML well-formedness and render rate vs size

From `evaluate.py` JSON outputs. Two grouped bars per size — XML rate and render rate (overall, not within-valid).

In [ ]:
available_sizes = [s for s in SIZES if evals[s] is not None]
if available_sizes:
    xml_rates    = [evals[s].get('xml_rate', np.nan)            for s in available_sizes]
    render_rates = [evals[s].get('render_rate_overall', np.nan) for s in available_sizes]
    perplexities = [evals[s].get('perplexity', np.nan)          for s in available_sizes]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    x = np.arange(len(available_sizes))
    w = 0.35
    ax1.bar(x - w/2, xml_rates,    w, label='XML well-formed')
    ax1.bar(x + w/2, render_rates, w, label='Renders (cairosvg)')
    ax1.set_xticks(x); ax1.set_xticklabels(available_sizes)
    ax1.set_ylabel('Rate'); ax1.set_ylim(0, 1)
    ax1.set_title('Sample validity vs model size')
    ax1.legend(); ax1.grid(alpha=0.3, axis='y')

    N_eval = [N_NON_EMB[s] for s in available_sizes]
    ax2.plot(N_eval, perplexities, 'o-')
    ax2.set_xscale('log'); ax2.set_yscale('log')
    ax2.set_xlabel('Non-embedding params $N$')
    ax2.set_ylabel('Test perplexity')
    ax2.set_title('Perplexity vs model size')
    ax2.grid(alpha=0.3, which='both')

    plt.tight_layout()
    plt.savefig(FIG_DIR / 'sample_quality.pdf')
    plt.show()
else:
    print('No eval JSONs found yet — run evaluate.py per size first.')

## 8. Summary table for the report

Combines fit parameters, final losses, and sample-quality metrics into a single dataframe ready for inclusion in `report/report.tex`.

In [ ]:
rows = []
for s in SIZES:
    row = {'config': s, 'N': N_NON_EMB[s]}
    row['SP_val_loss']  = sp_finals.get(s, np.nan)
    row['µP_val_loss']  = mup_finals.get(s, np.nan)
    if evals[s] is not None:
        row['perplexity']  = evals[s].get('perplexity')
        row['xml_rate']    = evals[s].get('xml_rate')
        row['render_rate'] = evals[s].get('render_rate_overall')
    rows.append(row)

summary_full = pd.DataFrame(rows).set_index('config')
summary_full

## Inferences on scaling behavior

*To be filled with empirical observations once runs complete. Discussion points:*

1. **Fitted exponent $\alpha$ on SVG vs prose.** Kaplan et al. report $\alpha \approx 0.076$ for natural language. SVG has heavier substring repetition — does $\alpha$ shift higher (more reducible loss) or lower (more irreducible structure)?
2. **Entropy floor $c$.** What does the fitted $c$ say about the irreducible per-token entropy of SVG? Compare to bits/token of the raw SVG distribution.
3. **SP vs µP.** Did the LR transfer cleanly across sizes under µP? How much per-size LR drift do we observe under SP?
4. **Extrapolation accuracy.** If we trained one held-out point, how close did the µP prediction come?
5. **Sample quality scaling.** Does XML well-formedness improve smoothly with $N$, or are there discrete jumps?

## 9. Tokenization statistics — sequence length histogram

Distribution of SVG-document token lengths (in BPE tokens). Computed from the train split by
splitting on the EOT token. The vertical line at `max_seq_len = 1024` marks the truncation
threshold used during training.


In [ ]:
import numpy as np, matplotlib.pyplot as plt

TRAIN_NPY = 'data/processed/train.npy'
TOKENIZER_DIR = 'tokenizers/bpe_4096'
from tokenizers import ByteLevelBPETokenizer
tok = ByteLevelBPETokenizer.from_file(f'{TOKENIZER_DIR}/vocab.json', f'{TOKENIZER_DIR}/merges.txt')
EOT_ID = tok.token_to_id('<|endoftext|>')
MAX_SEQ_LEN = 1024

arr = np.load(TRAIN_NPY)
boundaries = np.where(arr == EOT_ID)[0]
lengths = np.diff(np.concatenate([[-1], boundaries]))   # tokens per document (incl. trailing EOT)
print(f'documents: {len(lengths):,}')
print(f'mean / median / max length: {lengths.mean():.0f} / {np.median(lengths):.0f} / {lengths.max()}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(lengths, bins=80, color='C0', alpha=0.85)
ax.axvline(MAX_SEQ_LEN, color='red', linestyle='--', label=f'max_seq_len = {MAX_SEQ_LEN}')
ax.set_xlabel('Document length (BPE tokens)')
ax.set_ylabel('Count')
ax.set_title('Token-length distribution of training documents')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'seq_len_histogram.pdf')
plt.show()


## 10. Rendered SVG examples — input distribution

Decode and render a handful of training documents at different lengths, so the report can show what
the data actually looks like. Saves a PNG grid to `report/figures/data_examples.png`.


In [ ]:
import io
from PIL import Image
import cairosvg

def decode_doc(arr, start, eot_id, tokenizer):
    end = start
    while end < len(arr) and arr[end] != eot_id:
        end += 1
    return tokenizer.decode(arr[start:end].tolist())

# Pick documents covering a range of lengths
starts = np.concatenate([[0], boundaries[:-1] + 1])
order  = np.argsort(lengths)
picks  = [order[len(order) * p // 100] for p in (10, 30, 50, 70, 90)]
examples = [(int(lengths[i]), decode_doc(arr, int(starts[i]), EOT_ID, tok)) for i in picks]

fig, axes = plt.subplots(1, len(examples), figsize=(3*len(examples), 3.2))
for ax, (L, svg) in zip(axes, examples):
    try:
        png = cairosvg.svg2png(bytestring=svg.encode('utf-8'), output_width=240)
        ax.imshow(Image.open(io.BytesIO(png)))
    except Exception as e:
        ax.text(0.5, 0.5, f'render fail\n{e.__class__.__name__}', ha='center', va='center')
    ax.set_title(f'{L} tokens', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(FIG_DIR / 'data_examples.png', dpi=150)
plt.show()


## 11. Throughput and peak GPU memory per size

Per-step tokens/sec and peak GPU memory taken from each run's `log.csv`. Useful for the report's
Methods section (compute footprint per size) and for sanity-checking that bigger models do scale
memory roughly with $N$.


In [ ]:
rows = []
for s in SIZES:
    df = sp_logs[s]
    if df is None or 'tokens_per_sec' not in df.columns:
        continue
    rows.append({
        'config': s,
        'N':      N_NON_EMB[s],
        'tokens_per_sec_med': df['tokens_per_sec'].median(),
        'peak_mem_gb_max':    df['peak_mem_gb'].max(),
    })
throughput = pd.DataFrame(rows).set_index('config')
throughput


In [ ]:
if not throughput.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(throughput['N'], throughput['tokens_per_sec_med'], 'o-')
    ax1.set_xscale('log'); ax1.set_yscale('log')
    ax1.set_xlabel('Non-embedding params $N$')
    ax1.set_ylabel('Tokens/sec (median)')
    ax1.set_title('Training throughput vs model size')
    ax1.grid(alpha=0.3, which='both')

    ax2.plot(throughput['N'], throughput['peak_mem_gb_max'], 'o-', color='C1')
    ax2.set_xscale('log')
    ax2.set_xlabel('Non-embedding params $N$')
    ax2.set_ylabel('Peak GPU memory (GB)')
    ax2.set_title('Peak GPU memory vs model size')
    ax2.grid(alpha=0.3, which='both')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'compute_footprint.pdf')
    plt.show()
else:
    print('No throughput data — runs predate the tokens_per_sec / peak_mem_gb columns.')


## 12. Render generated samples — qualitative grid

For the **best** model, sample N times and render to PNG. The PDF requires this in the report's
Results section.

**Prerequisite:** run `generate.py` against your best checkpoint with `--output_dir samples/best/`
first, then this cell renders whatever's in that directory.


In [ ]:
from pathlib import Path
import io
from PIL import Image
import cairosvg

SAMPLE_DIR = Path('samples/best')
if SAMPLE_DIR.exists():
    files = sorted(SAMPLE_DIR.glob('sample_*.svg'))[:9]
    if files:
        n = len(files)
        cols = 3
        rows = (n + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows))
        axes = np.array(axes).reshape(-1)
        for i, fp in enumerate(files):
            ax = axes[i]
            with open(fp) as f:
                svg = f.read()
            try:
                png = cairosvg.svg2png(bytestring=svg.encode('utf-8'), output_width=240)
                ax.imshow(Image.open(io.BytesIO(png)))
                ax.set_title(fp.stem, fontsize=9)
            except Exception as e:
                ax.text(0.5, 0.5, f'render fail', ha='center', va='center')
                ax.set_title(fp.stem, fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
        for ax in axes[n:]:
            ax.axis('off')
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'generated_samples.png', dpi=150)
        plt.show()
    else:
        print(f'No sample_*.svg files in {SAMPLE_DIR}')
else:
    print(f'{SAMPLE_DIR} does not exist — run generate.py with --output_dir first.')
